# 한국어 최상급의 위치를 바꾸면 모델의 답도 달라질까?

조형준 | 서울대학교 언어학과

한국어 최상급 표현에 대한 LLM 추론 평가: 문항 생성 및 실험 결과 재현


1. Lee의 예문에서 ‘논문·읽다’를 가져와 어휘표의 첫 행을 만들고, 이 행에서 전제·가설 네 조합을 만든다.
2. 그중 한 문항을 모델에 어떻게 물었는지 보고, 실제 응답을 채점한다.


## 1. 살펴볼 문제

- 관형형:　철수가 **가장 많은 논문을** 읽었다.
- 부사형:　철수가 **논문을 가장 많이** 읽었다.

“누가 논문을 제일 많이 읽었어?”라고 물으면 두 문장 모두 답이 된다. \
“철수는 어떤 글을 제일 많이 읽었어?”라고 물으면 두 번째 문장이 자연스럽다. 

Lee(2018)는 첫 번째 문장에서는 글의 종류끼리 비교하는 해석이 어렵다고 보았다. 모델에도 전제와 가설을 한 문장씩 주고 이 차이가 답에 드러나는지 살펴봤다. [Lee의 예문 (8), (10)](https://web.stanford.edu/group/cslipublications/cslipublications/ja-ko-contents/JK25/JK25-Lee.pdf#page=4)

### 어휘표를 만들 때 무엇을 정했나?

Lee의 예문에서 가져온 어휘는 ‘논문’과 ‘읽다’다. 문항을 늘리면서 같은 동사에 ‘기사, 보고서, 편지’를 넣었다./
아래는 당시 생성 코드의 수량 어휘 목록에서 다섯가지의 조합을 가져온 것이다.

In [1]:

words = [("논문", "read"), ("기사", "read"), ("보고서", "read"),
         ("편지", "read"), ("우표", "collect")]
sample_rows = []

for number, (noun, verb) in enumerate(words, start=1):
    row = {"group": f"Q{number:02d}", "family": "quantity", "noun": noun,
           "stage": "기본", "verb_family": verb,
           "category": "글" if verb == "read" else "수집품"}
    sample_rows.append(row)

for row in (sample_rows[0], sample_rows[-1]):
    print(row["group"], row["noun"], row["verb_family"], row["category"])

Q01 논문 read 글
Q05 우표 collect 수집품


In [2]:
sample_rows

[{'group': 'Q01',
  'family': 'quantity',
  'noun': '논문',
  'stage': '기본',
  'verb_family': 'read',
  'category': '글'},
 {'group': 'Q02',
  'family': 'quantity',
  'noun': '기사',
  'stage': '기본',
  'verb_family': 'read',
  'category': '글'},
 {'group': 'Q03',
  'family': 'quantity',
  'noun': '보고서',
  'stage': '기본',
  'verb_family': 'read',
  'category': '글'},
 {'group': 'Q04',
  'family': 'quantity',
  'noun': '편지',
  'stage': '기본',
  'verb_family': 'read',
  'category': '글'},
 {'group': 'Q05',
  'family': 'quantity',
  'noun': '우표',
  'stage': '기본',
  'verb_family': 'collect',
  'category': '수집품'}]

Q01은 ‘논문을 읽다’, Q05는 ‘우표를 모으다’에 붙인 번호이다.

두 문항 모두 개수를 비교하므로 `family`는 `quantity`.

논문은 “다른 종류의 **글**보다”, 우표는 “다른 종류의 **수집품**보다”라고 쓴다. 이 자리에 들어갈 단어를 `category`에 기록했다.

| 항목 | 방법 | 어휘표에 들어 있는 값 |
|---|---|---|
| `family` | 어휘에 맞는 문장 틀을 고른다. | 수량(`quantity`), 구매·선물 가격(`price_gift`), 소유(`possession`), 글의 길이(`text_length`), 그림의 크기(`drawing_size`), 구조물의 높이(`construction_height`) |
| `category` | 수량 가설의 “다른 종류의 ___보다”에 넣을 단어를 정한다. | `글` 10묶음, `수집품` 10묶음. |
| `stage` | 어휘를 어느 단계에서 추가했는지 기록. | `기본` 50묶음, `척도 확장` 15묶음, `후속 증강` 45묶음 |

다음 셀에서는 방금 만든 다섯 행이 실제 실험에 사용한 어휘표 `data/lexicon.json`의 해당 행과 일치하는지 확인한다.


| 단계 | 실제 예시 | 달라진 점 |
|---|---|---|
| **척도 확장** | ‘논문을 가장 **많이** 읽었다’ → ‘편지를 가장 **길게** 썼다’ | 개수나 가격을 비교하던 자료에 **글의 길이, 그림의 크기, 구조물의 높이**를 비교하는 문장 틀을 추가했어. |
| **후속 증강** | ‘**편지**를 가장 길게 썼다’ → ‘**수필**을 가장 길게 썼다’ | 비교 기준은 여전히 **글의 길이**야. 이미 만든 틀에 ‘수필’ 같은 새 어휘를 넣어 문항을 늘렸어. |

In [3]:
# 이제 실험에 쓴 전체 어휘표와 척도표를 읽는다.

import json
from pathlib import Path
import pandas as pd
from benchmark import particle, construction_templates, item_row
from experiment import ROUTES, request_body, send_one

data = Path("data") # 이미 만든 데이터를 불러오기
def read_json(name):
    return json.loads((data / name).read_text(encoding="utf-8"))

def read_lines(name):
    with (data / name).open(encoding="utf-8") as file:
        return [json.loads(line) for line in file if line.strip()]

slots = read_json("lexicon.json")
scales = read_json("scale_forms.json")
slot_by_id = {s["group"]: s for s in slots}

# 앞에서 만든 다섯 행이 저장된 어휘표와 같은지 확인!
for row in sample_rows: 
    assert row == slot_by_id[row["group"]]
print("전체 어휘 묶음:", len(slots))

전체 어휘 묶음: 110


In [4]:
slots[0]

{'group': 'Q01',
 'family': 'quantity',
 'noun': '논문',
 'stage': '기본',
 'verb_family': 'read',
 'category': '글'}

In [5]:
scales

{'text_length': {'adnom': '긴',
  'adv': '길게',
  'adj': '길었다',
  'past': '썼다',
  'rel': '쓴',
  'ben': '써주었다',
  'brel': '써준',
  'category': '글'},
 'drawing_size': {'adnom': '큰',
  'adv': '크게',
  'adj': '컸다',
  'past': '그렸다',
  'rel': '그린',
  'ben': '그려주었다',
  'brel': '그려준',
  'category': '도형'},
 'construction_height': {'adnom': '높은',
  'adv': '높게',
  'adj': '높았다',
  'past': '쌓았다',
  'rel': '쌓은',
  'ben': '쌓아주었다',
  'brel': '쌓아준',
  'category': '구조물'},
 'gift': {'adnom': '비싼',
  'adv': '비싸게',
  'adj': '비쌌다',
  'past': '샀다',
  'rel': '산',
  'ben': '사주었다',
  'brel': '사준',
  'category': '선물'}}

text_length는 글의 길이를 비교하는 문항에 쓰는 표현 묶음

| 문장 틀 | scales |
|---|---|
| 철수가 가장 ___ 편지를 썼다. | `긴` |
| 철수가 편지를 가장 ___ 썼다. | `길게` |
| 철수가 영희에게 써준 편지는 다른 편지보다 ___. | `길었다` |

## 2. ‘논문을 읽다’로 네 문항 만들기

### benchmark.py에서 무엇을 고정했나?

‘가장 많은 논문을 읽었다’와 ‘논문을 가장 많이 읽었다’에 **똑같은 가설**을 붙여야 구조를 바꾼 효과를 비교할 수 있다.

그래서 전제를 만드는 부분과 가설을 만드는 부분을 나눴다.

아래는 수량 문항을 만드는 함수. 첫번째 사전에는 전제가, 두번째 사전에는 가설이 들어간다.

In [6]:
# premises에는 두 구조가, hypotheses에는 두 비교 대상이 들어 있다.

import inspect # 파일에 정의된 함수의 소스 코드를 볼 수 있게 해준다.
from benchmark import quantity_sentences # 비교 가능한 범주 중에 quantity

print(inspect.getsource(quantity_sentences)) # quantity_sentences에 있는 코드


def quantity_sentences(slot):
    """수량 문항 한 묶음의 전제 두 개와 가설 두 개를 만든다."""
    noun = particle(slot["noun"], "을", "를")
    past, relative = ("읽었다", "읽은") if slot["verb_family"] == "read" else ("모았다", "모은")
    premises = {
        "nominal": f"철수가 가장 많은 {noun} {past}.",
        "adverbial": f"철수가 {noun} 가장 많이 {past}.",
    }
    hypotheses = {
        "category": f"철수는 자신이 {relative} 다른 종류의 {slot['category']}보다 {noun} 더 많이 {past}.",
        "person": f"철수는 다른 사람들보다 {noun} 더 많이 {past}.",
    }
    return premises, hypotheses



철수는 자신이 {relative} 다른 종류의 {slot['category']}보다 {noun} 더 많이 {past}.

읽은, 글, 논문을, 읽었다

In [7]:
slot = slot_by_id["Q01"]
if slot["verb_family"] == "read":
    past = "읽었다"
    relative = "읽은"
else:
    past = "모았다"
    relative = "모은"

이제 함수를 실행한다! 출력에서 ‘가장 많은 논문을’과 ‘논문을 가장 많이’의 위치를 비교해 보자.

In [8]:
# Q01을 함수에 넣어 모델들이 해결한 문항이 될 네 문장을 출력한다.
# 위 두 줄은 전제, 아래 두 줄은 같은 전제에 각각 붙일 가설이다.

group = "Q01" 
premises, hypotheses = quantity_sentences(slot_by_id[group]) #앞서 확인한 함수를 호출

# slot_by_id[group] -> 여기서 꺼낸 ‘논문, 읽다, 글’ 정보를 quantity_sentences()에 넘긴다.

for form, sentence in premises.items():
    print(form, sentence)
for target, sentence in hypotheses.items():
    print(target, sentence)

nominal 철수가 가장 많은 논문을 읽었다.
adverbial 철수가 논문을 가장 많이 읽었다.
category 철수는 자신이 읽은 다른 종류의 글보다 논문을 더 많이 읽었다.
person 철수는 다른 사람들보다 논문을 더 많이 읽었다.


| 코드에서 붙인 이름 | 뜻 | 만들어진 문장 |
|---|---|---|
| nominal | 전제 ① 관형형 | 철수가 **가장 많은 논문을** 읽었다. |
| adverbial | 전제 ② 부사형 | 철수가 **논문을 가장 많이** 읽었다. |
| category | 가설 ① 글 종류 비교 | 철수는 자신이 읽은 **다른 종류의 글보다** 논문을 더 많이 읽었다. |
| person | 가설 ② 사람 비교 | 철수는 **다른 사람들보다** 논문을 더 많이 읽었다. |

In [9]:
# 전제 두 개와 가설 두 개를 조합하고 각 조합에 정답을 붙인다.
# 표에서 글 종류를 비교한 두 줄의 정답을 먼저 읽는다. 

def quantity_answer(form, target):
    if form == "nominal" and target == "category":
        return "neutral"
    return "entailment"

form_ko = {"nominal": "관형형", "adverbial": "부사형", "predicative": "서술형"}
target_ko = {"category": "종류끼리", "person": "사람끼리"}

pd.DataFrame([
    {"구조": form_ko[f], "전제": p, "비교": target_ko[t], "가설": h, "정답안": quantity_answer(f, t)}
    for t, h in hypotheses.items() for f, p in premises.items()
])

,구조,전제,비교,가설,정답안
0,관형형,철수가 가장 많은 논문을 읽었다.,종류끼리,철수는 자신이 읽은 다른 종류의 글보다 논문을 더 많이 읽었다.,neutral
1,부사형,철수가 논문을 가장 많이 읽었다.,종류끼리,철수는 자신이 읽은 다른 종류의 글보다 논문을 더 많이 읽었다.,entailment
2,관형형,철수가 가장 많은 논문을 읽었다.,사람끼리,철수는 다른 사람들보다 논문을 더 많이 읽었다.,entailment
3,부사형,철수가 논문을 가장 많이 읽었다.,사람끼리,철수는 다른 사람들보다 논문을 더 많이 읽었다.,entailment


### 900문항을 만든 과정

| 추가한 문항 | 문항 수 |
|---|---:|
| 수량, 구매, 선물, 소유 | 280 |
| 글의 길이, 도형의 크기, 구조물의 높이 | 105 |
| ‘영희에게 써주었다’와 같은 문장의 부사형 짝 | 105 |
| 새 어휘 45묶음으로 만든 문항 | 410 |
| 합계 | 900 |

‘영희에게 편지를 써주었다’에서는 글을 쓴 사람, 글을 받은 사람, 글의 종류를 각각 비교할 수 있는지 물었다.

소유 구문은 ‘가장 많은 …이 있다’와 ‘…이 가장 많다’를 짝지었다.

또한 단어를 바꿀 때는 두 문장의 비교 기준이 같은지 살폈다. (예를 들어 ‘책을 높게 쌓았다’에서 높은 것은 책 더미인데, ‘높은 책’은 한 권의 높이로 읽혀서 제외.) 

In [10]:
# 어휘 묶음마다 맞는 문장 틀을 골라 전제+가설 한 조합을 한 행에 넣는다.

# 어휘 묶음으로 서로 다른 문항을 만든다.

def proposed_answer(form, target):
    if form == "nominal" and target in {"category", "product", "band"}:
        return "neutral"
    return "entailment"


def build_benchmark(slots):
    rows = []
    for slot in slots: # 어휘를 한 묶음씩 꺼낸다
        frames = construction_templates(slot, scales, quantity_sentences) # 묶음을 가지고 전제와 가설을 만든다.
        for module, benefactive, (premises, hypotheses) in frames:
            for target, hypothesis in hypotheses.items(): # 가설 하나에 대해 전제를 붙이기 # target은 가설의 이름이고, hypothesis는 실제 가설 문장.
                for form, premise in premises.items(): # form은 전제의 구조 이름이고, premise는 실제 전제 문장
                    key = proposed_answer(form, target) # 정답안을 붙이고 문항 하나를 저장
                    rows.append(item_row(slot, module, benefactive, target,
                                         form, premise, hypothesis, key))
    return pd.DataFrame(rows)

items = build_benchmark(slots)
print(f"어휘 {len(slots)}묶음, {len(items)}문항, 대립쌍 {items.pair_id.nunique()}개")

어휘 110묶음, 900문항, 대립쌍 450개


target = "category"

hypothesis = "철수는 자신이 읽은 다른 종류의 글보다 논문을 더 많이 읽었다."


form = "nominal"

premise = "철수가 가장 많은 논문을 읽었다."

In [11]:
# 같은 짝의 두 정답이 같으면 통제쌍, 다르면 진단쌍이다.

answer_count = items.groupby("pair_id")["key"].nunique()
pair_type = answer_count.map({1: "통제", 2: "진단"})
print(pair_type.value_counts().to_string())
assert pair_type.value_counts().to_dict() == {"진단": 200, "통제": 250}

key
통제    250
진단    200


## 3. 모델에게 물어보기

### 한국어 지시문과 영어 지시문

전제와 가설은 항상 한국어로 제시했다. 매번 새 대화에 한 문항만 넣고 보기 세 개 중 하나를 고르도록 했다. 아래 셀의 K1을 E1으로 바꾸면 같은 문항의 영어 지시문을 볼 수 있다.

In [12]:
prompts = read_json("prompts.json") 

def make_messages(item, condition):
    template = prompts[condition]
    question = template["body"].format(premise=item.premise, hypothesis=item.hypothesis)

    text = "\n\n".join([template["common"], question, template["output"]]) #질문 앞뒤에 지시를 붙여 출력
    return [{"role": "user", "content": text}]

item_id = "CORE-Q01-ADV"
condition = "K1"  # 영어 지시문은 "E1"
by_id = items.set_index("id")
messages = make_messages(by_id.loc[item_id], condition)
print(messages[0]["content"])

전제가 참이라고 가정하고 다음 기준으로 가설을 분류하세요.
entailment: 전제의 언어적으로 허용되는 해석 중 적어도 하나에서, 전제가 참인 모든 상황에 가설도 참이다.
contradiction: 전제의 언어적으로 허용되는 모든 해석에서, 전제가 참인 모든 상황에 가설은 거짓이다.
neutral: 위 두 조건에 해당하지 않는다.
가설이 참일 수 있다는 것만으로 entailment를 선택하지 마세요. 전제에 없는 사실을 추가하지 말고, 전제와 가설에서 척도·참여자·문맥상 비교 대상을 일관되게 유지하세요.

전제: 철수가 논문을 가장 많이 읽었다.
가설: 철수는 자신이 읽은 다른 종류의 글보다 논문을 더 많이 읽었다.

위 기준에 따라 전제와 가설의 관계를 판단하세요.

entailment, contradiction, neutral 중 하나만 출력하세요.


### 모델당 1,800건

900문항에 지시문 두 종류를 붙이고 실험 당시 순서대로 배열했다. 한국어가 주요 조건이고 영어는 지시 언어를 바꿔 보는 조건이다.

## 4. 응답 채점하기

앞뒤 공백을 지운 뒤 **첫 줄 전체가 라벨 하나**이면 대소문자에 관계없이 인정.


In [13]:
# 응답 첫 줄에서 라벨을 읽어 문항 번호로 정답표에 연결한다.

def first_label(text):
    lines = (text or "").strip().splitlines() # 공백, 즐 전처리
    first = lines[0].strip().lower() if lines else "" # lines[0] 응답으로 예상되는 단어 꺼내고 소문자
    return first if first in {"entailment", "neutral", "contradiction"} else "invalid"

answers = pd.DataFrame(read_lines("responses.jsonl")) # 모든 응답에 같은 규칙을 적용
answers["answer"] = answers.response_text.map(first_label)

answers = answers.merge(items[["id", "pair_id", "structure", "comparison_target", "key"]], 
                        left_on="item_id", right_on="id", validate="many_to_one") # 문항 번호로 정답을 찾아 채점
answers["match"] = answers.answer.eq(answers.key)

In [14]:
example = answers[(answers["model"] == "upstage/solar-pro4") & (answers.condition == "K1")
                  & answers.item_id.isin(["CORE-Q01-NOM", "CORE-Q01-ADV",
                                         "CONTROL-Q01-NOM", "CONTROL-Q01-ADV"])]

view = example.sort_values(["pair_id", "structure"], ascending=[True, False])

pd.DataFrame({
    "비교 대상": view.comparison_target.map(target_ko),
    "전제": view.structure.map(form_ko),
    "정답안": view.key,
    "응답": view.answer,
    "채점": view.match.map({True: "일치", False: "불일치"}),
}).reset_index(drop=True)

,비교 대상,전제,정답안,응답,채점
0,종류끼리,관형형,neutral,entailment,불일치
1,종류끼리,부사형,entailment,entailment,일치
2,사람끼리,관형형,entailment,entailment,일치
3,사람끼리,부사형,entailment,entailment,일치


Solar는 네 문항에 전부 함의라고 답했다.

In [15]:
# 모델 하나가 지시문 한 조건에서 낸 응답 900건을 채점한다.
def score_pairs(rows):
    # 같은 짝의 두 문항이 모두 맞아야 True가 된다.
    pair_correct = rows.groupby("pair_id")["match"].all()

    # 앞에서 만든 pair_type으로 진단쌍과 통제쌍을 나누어 센다.
    counts = pair_correct.groupby(pair_type).sum()

    return {
        "개별 /900": int(rows["match"].sum()),
        "진단 /200쌍": int(counts["진단"]),
        "통제 /250쌍": int(counts["통제"]),
        "무효": int(rows["answer"].eq("invalid").sum()),
    }


# 모델과 지시문별로 결과 한 행씩 만든다.
records = []

for (model_name, condition), rows in answers.groupby(["model", "condition"]):
    record = {"모델": model_name, "지시문": condition}
    record.update(score_pairs(rows))
    records.append(record)

results = pd.DataFrame(records)

## 5. 결과

In [16]:
full_table = results.set_index(["모델", "지시문"]).copy()

# 두 문항을 모두 맞힌 쌍의 비율을 유형별로 계산
full_table["진단쌍 일치율(%)"] = (full_table["진단 /200쌍"] / 200 * 100).round(1)
full_table["통제쌍 일치율(%)"] = (full_table["통제 /250쌍"] / 250 * 100).round(1)

# 각 유형에서 몇 쌍을 맞혔고, 전체의 몇 퍼센트인지
full_table[["진단 /200쌍", "진단쌍 일치율(%)", "통제 /250쌍", "통제쌍 일치율(%)"]]

진단 /200쌍  진단쌍 일치율(%)  통제 /250쌍  \
모델                                지시문                                   
anthropic/claude-sonnet-4.5       E1          0         0.0        68   
                                  K1          5         2.5        79   
gpt-4o-2024-08-06                 E1         33        16.5       197   
                                  K1         28        14.0        85   
gpt-5.2-2025-12-11                E1         27        13.5       118   
                                  K1         10         5.0        64   
gpt-6.1-terra                     E1         95        47.5        86   
                                  K1         88        44.0       102   
kt/midm-k-3.0-pro                 E1         15         7.5       150   
                                  K1         28        14.0       173   
meta-llama/llama-3.1-70b-instruct E1         16         8.0       233   
                                  K1         25        12.5       244   
qwen/qwen-2.5-72b-instruct        E1         28        14.0       154   
                                  K1          8         4.0       149   
upstage/solar-pro4                E1         32        16.0       110   
                                  K1         33        16.5       110   

                                       통제쌍 일치율(%)  
모델                                지시문              
anthropic/claude-sonnet-4.5       E1         27.2  
                                  K1         31.6  
gpt-4o-2024-08-06                 E1         78.8  
                                  K1         34.0  
gpt-5.2-2025-12-11                E1         47.2  
                                  K1         25.6  
gpt-6.1-terra                     E1         34.4  
                                  K1         40.8  
kt/midm-k-3.0-pro                 E1         60.0  
                                  K1         69.2  
meta-llama/llama-3.1-70b-instruct E1         93.2  
                                  K1         97.6  
qwen/qwen-2.5-72b-instruct        E1         61.6  
                                  K1         59.6  
upstage/solar-pro4                E1         44.0  
                                  K1         44.0